# Chapter 28 — Context Admission Is Application Policy

**Companion to *Pi Agents*** · [`Pi Agents` chapter 28](https://programmer.ie/books/pi/28-chapter/)

| | |
|---|---|
| Chapter | `28-chapter.md` · weight 28 · `/books/pi/28-chapter/` |
| Notebook | `notebooks/pi/28-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Starting in `repo/src/billing`, which `AGENTS.md` files are discovered and
admitted — and does an override reach outside its directory?

The chapter's point: discovery walks the **working directory and its parents**,
plus the agent directory. Siblings are never discovered. An `AGENTS.override.md`
replaces the `AGENTS.md` beside it and **only that one**. And a host can drop or
add files before they reach the model — admission is application policy.


## What this notebook establishes

- Discovery walks the **agent directory**, the **working directory** and its **parents** — a sibling directory is never discovered.
- An `AGENTS.override.md` **replaces the `AGENTS.md` beside it**, and only that one — it does not suppress other directories.
- The host **owns admission**: an `agentsFilesOverride` can drop a file before it reaches the model, or add a virtual one.
- The merge order of several context files at once is **not specified** by Pi (`ch28-lim1`).
- Whether the **model received** the files was not run (`ch28-lim3`) — this is the loader's admitted list, not a provider request.

## What this notebook does **not** establish

- **The merge order of several context files at once is not specified** (`ch28-lim1`).
- **Whether the model received the files** was not run (`ch28-lim3`) — this reads the loader, not the provider.
- **Admission is application policy**, not a Pi guarantee — the `agentsFilesOverride` is a hook the host uses.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — `DefaultResourceLoader` built directly over a real fixture tree on disk.
* **Evidence scope:** `in_process_runtime`. The discovery rules and the override hook are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(28))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/28-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: what is discovered from `repo/src/billing`

In [3]:
ADM = pinb.driver_source("ch28-admission.ts")
adm = pinb.driver(ADM, label="ch28-admission", timeout=180)
print("Admitted files:")
for f in adm["discovered"]["list"]:
    print("  -", f)
print()
pinb.show_checks([
    pinb.check("the agent directory applies everywhere",
               adm["discovered"]["includesUser"], "user rules present"),
    pinb.check("a parent directory is inherited downward",
               adm["discovered"]["includesRepo"], "repo rules present"),
    pinb.check("a sibling directory is never discovered",
               not adm["discovered"]["includesDocsSibling"], "docs rules absent"),
])

Admitted files:
  - AGENTS.md=user rules
  - AGENTS.md=repo rules
  - AGENTS.override.md=billing override

== Assertions ==
  assertion  observed
----  ----------------------------------------  ------------------
PASS  the agent directory applies everywhere  user rules present
PASS  a parent directory is inherited downward  repo rules present
PASS  a sibling directory is never discovered  docs rules absent

3/3 assertions held.


## An override replaces the file beside it, and only that one

In [4]:
print("Admitted files with the override:")
for f in adm["override"]["list"]:
    print("  -", f)
print()
pinb.show_checks([
    pinb.check("the override is admitted",
               adm["override"]["hasOverride"], "billing override present"),
    pinb.check("the AGENTS.md beside it is replaced",
               not adm["override"]["hasReplaced"], "billing rules (replaced) absent"),
    pinb.check("an override does not suppress other directories",
               adm["override"]["hasRepoStill"], "repo rules still present"),
])

Admitted files with the override:
  - AGENTS.md=user rules
  - AGENTS.md=repo rules
  - AGENTS.override.md=billing override

== Assertions ==
  assertion  observed
----  -----------------------------------------------  -------------------------------
PASS  the override is admitted  billing override present
PASS  the AGENTS.md beside it is replaced  billing rules (replaced) absent
PASS  an override does not suppress other directories  repo rules still present

3/3 assertions held.


## The host owns admission: drop and add

In [5]:
print("Admitted files after the host override:")
for f in adm["host"]["list"]:
    print("  -", f)
print()
pinb.show_checks([
    pinb.check("the host can drop a file before it reaches the model",
               not adm["host"]["hasUserRules"], "user rules dropped"),
    pinb.check("the host can add a virtual file",
               adm["host"]["hasVirtual"], "host policy added"),
])

Admitted files after the host override:
  - AGENTS.md=repo rules
  - AGENTS.override.md=billing override
  - AGENTS.md=policy supplied by the host

== Assertions ==
  assertion  observed
----  ----------------------------------------------------  ------------------
PASS  the host can drop a file before it reaches the model  user rules dropped
PASS  the host can add a virtual file  host policy added

2/2 assertions held.


## The chapter's own tests

All three admission tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [6]:
PATTERNS = 'ch28-admission/admission.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch28-admission/admission.test.ts', show="admitted")

$ node --test ch28-admission/admission.test.ts
  PASS  3 passed, 0 failed, 0 skipped  in 874 ms

  tests matching 'admitted':
    ok   admitted: an override replaces the AGENTS.md beside it, and only that one


## Your turn: predict, then run

**Predict first.** Starting in `repo/docs` instead of `repo/src/billing`, which
files would be discovered? Does `billing`'s override come along?

**Then change one input.** Move the override to `repo/AGENTS.override.md` (the
repository root). What replaces what?

**Predict the boundary.** The chapter says "the merge order of several context
files at once is not specified." If the repo root and the project both define a
rule with the same name, which wins?

In [7]:
print("Predictions:")
print("  1. From repo/docs: repo rules + user rules; billing is a sibling, not discovered.")
print("  2. Override at repo root: replaces repo/AGENTS.md, not billing/.")
print("  3. Same-name rule: the merge order is unspecified - do not rely on it.")
print()
print("Observed above:")
user = adm["discovered"]["includesUser"]
repo = adm["discovered"]["includesRepo"]
docs = adm["discovered"]["includesDocsSibling"]
print(f"  discovered: user={user}, repo={repo}, docs sibling={docs}")
print(f"  override: present={adm['override']['hasOverride']}, replaced={adm['override']['hasReplaced']}, repo still={adm['override']['hasRepoStill']}")
print(f"  host: user dropped={not adm['host']['hasUserRules']}, virtual added={adm['host']['hasVirtual']}")
print()
assert adm["discovered"]["includesUser"] and adm["discovered"]["includesRepo"]
assert not adm["discovered"]["includesDocsSibling"]
assert adm["override"]["hasOverride"] and not adm["override"]["hasReplaced"] and adm["override"]["hasRepoStill"]
assert not adm["host"]["hasUserRules"] and adm["host"]["hasVirtual"]
print("held: discovery walks up, not sideways; an override replaces its neighbour only; the host can drop and add")

Predictions:
  1. From repo/docs: repo rules + user rules; billing is a sibling, not discovered.
  2. Override at repo root: replaces repo/AGENTS.md, not billing/.
  3. Same-name rule: the merge order is unspecified - do not rely on it.

Observed above:
  discovered: user=True, repo=True, docs sibling=False
  override: present=True, replaced=False, repo still=True
  host: user dropped=True, virtual added=True

held: discovery walks up, not sideways; an override replaces its neighbour only; the host can drop and add


## Interpretation

Context admission has three layers, and the chapter shows the boundaries:

| Layer | What it decides | The boundary |
|---|---|---|
| **Discovery** | Which files are candidates | Working dir + parents + agent dir; **not siblings** |
| **Admission** | Which candidates load | An `AGENTS.override.md` replaces its neighbour **only** |
| **Host override** | What the model finally sees | `agentsFilesOverride` can **drop or add** files |

Practical rules:
1. **Context comes from the path, not the project** — a file is loaded because it is on the path from the working directory to the root.
2. **An override is local** — it replaces one file, not a directory's worth.
3. **The host can filter** — `agentsFilesOverride` is where an application enforces its own policy.
4. **Merge order is unspecified** — do not depend on which of two same-name rules wins.
5. **This is admission, not delivery** — whether the model received the files is a separate question (chapter 5's stored-vs-sent).

## Sources, execution mode and limitations

**Execution mode:** **Run** — `DefaultResourceLoader` built directly over a real fixture tree on disk.

**Evidence scope:** `in_process_runtime`. The discovery rules and the override hook are Pi's.

### What was read or run

- **Ran** `drivers/ch28-admission.ts`: three real `DefaultResourceLoader` builds over a fixture tree (discovery, override locality, host admission).
- **Ran** `ch28-admission/admission.test.ts` (3 tests).
- **Read** `examples/evidence.json`, chapter 28 rows (3 claims).

### Limitations

- **The merge order of several context files at once is not specified** (`ch28-lim1`).
- **Whether the model received the files** was not run (`ch28-lim3`) — this reads the loader, not the provider.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
